# 60.04 Сценарный расчёт эквивалентной сферы, УО и ФВ

**Статус результата:** `exploratory_hypothesis_not_validated`.

Ноутбук доводит исследовательскую цепочку до численных сценариев объёма,
ударного объёма и фракции выброса. Он показывает, какие значения следуют из
кандидатного остатка ТТРКГ при разных неизвестных коэффициентах оператора
«безразмерный сигнал → объём».

Входной остаток хранится на нормированной фазовой шкале `phase_rr`; эта шкала
не трактуется как время после R. Расчёт не переводится в строгие результаты
УО/ФВ, потому что целевой желудочек, механические фазы, оператор объёма, его
калибровка и ковариация отсутствуют.

## Сильные допущения вычислительного эксперимента

Для каждой остаточной кривой временно предполагается:

1. максимум и минимум ансамблевой кривой соответствуют ED и ES одного цикла;
2. размах остатка линейно связан с изменением объёма коэффициентом $G_V$;
3. изменение эквивалентной сферы можно приравнять УО выбранного желудочка;
4. выбранное значение EDV относится к тому же желудочку и циклу;
5. приборная шкала, кандидатная разметка и тканевая коррекция предыдущих
   этапов работают в заданных сценариях.

Ни одно из этих допущений пока не принято. $G_V$ перебирается от
$10^{-6}$ до $10^{-3}$ на миллилитр, EDV — от 50 до 250 мл. Эти сетки служат
только для проверки зависимости ответа и не являются референсными диапазонами.

Комбинации с `ESV < 0` отмечаются как физически недопустимые и не обрезаются.

In [1]:
# Загрузка исследовательского остатка 40.22
import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from exploratory_analysis import EXPLORATORY_STATUS, equivalent_volume_scenarios

EXP02_CONFIG_PATH = Path(os.environ["KALMYKOV_EXP02_CONFIG"]).expanduser().resolve()
CONFIG = json.loads(EXP02_CONFIG_PATH.read_text(encoding="utf-8"))
DERIVED = Path(CONFIG["derived_root"]).expanduser().resolve()
INPUT_PATH = DERIVED / "cross_experiment" / "exploratory" / "40.22_ttrkg_transfer_identifiability_exploratory.json"
INPUT = json.loads(INPUT_PATH.read_text(encoding="utf-8"))
if INPUT.get("status") != EXPLORATORY_STATUS or INPUT.get("strict_pipeline_authorized") is not False:
    raise RuntimeError("60.04 принимает только отдельный исследовательский артефакт 40.22")
if INPUT.get("analysis_scope") != "phase_aligned_ttrkg_vs_3304_conditional_side_scenarios":
    raise RuntimeError("60.04 ожидает фазовый артефакт 40.22 новой схемы")
OUT_DIR = DERIVED / "cross_experiment" / "exploratory"
OUT_PATH = OUT_DIR / "60.04_equivalent_sphere_sv_ef_scenarios.json"
VOLUME_SENSITIVITY_PER_ML = np.logspace(-6, -3, 7)
EDV_GRID_ML = np.asarray([50.0, 100.0, 150.0, 200.0, 250.0])


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

In [2]:
# Перебор всех остаточных сценариев экспериментов 2 и 3
results = []


def add_group(experiment_id, subject_id, mode_name, phase_rr, scenarios):
    for scenario in scenarios:
        volume = equivalent_volume_scenarios(
            scenario["residual_fractional"],
            VOLUME_SENSITIVITY_PER_ML,
            EDV_GRID_ML,
        )
        rows = volume["scenarios"]
        results.append({
            "experiment_id": experiment_id,
            "subject_id": subject_id,
            "mode": mode_name,
            "s_soft": scenario["s_soft"],
            "s_lung": scenario["s_lung"],
            "scenario_index_33_04": scenario["scenario_index_33_04"],
            "h_m": scenario["h_m"],
            "phase_rr": phase_rr,
            "residual_peak_to_peak": volume["residual_peak_to_peak"],
            "volume_scenarios": rows,
            "physically_admissible_count": sum(row["physically_admissible"] for row in rows),
            "scenario_count": len(rows),
        })


for subject_id, subject in INPUT["exp02"]["subjects"].items():
    for mode_name, mode in subject["modes"].items():
        add_group(
            "exp02",
            subject_id,
            mode_name,
            mode["phase_rr"],
            mode["transfer_scenarios"],
        )

for mode_name, mode in INPUT["exp03"]["main_record"]["modes"].items():
    add_group(
        "exp03",
        "exp03_nik",
        mode_name,
        mode["phase_rr"],
        mode["transfer_scenarios"],
    )

In [3]:
# Сохранение сценарной таблицы без статусов строгой серии 60
artifact = {
    "schema_version": 2,
    "status": EXPLORATORY_STATUS,
    "artifact_id": "60.04_equivalent_sphere_sv_ef_scenarios",
    "analysis_scope": "phase_rr_residual_volume_scenarios",
    "execution_branch": "real_data_exploratory_upstream_40_22",
    "strict_pipeline_authorized": False,
    "strict_60_01_status_emitted": False,
    "strict_60_02_status_emitted": False,
    "chamber_id": None,
    "phase_operator_status": "not_supplied",
    "volume_operator_status": "scenario_grid_not_calibrated",
    "input_provenance": {
        "upstream_artifact": INPUT_PATH.name,
        "upstream_sha256": sha256_file(INPUT_PATH),
        "upstream_status": INPUT.get("status"),
        "upstream_scope": INPUT.get("analysis_scope"),
        "configuration_sha256": sha256_file(EXP02_CONFIG_PATH),
    },
    "scenario_axes": {
        "phase_axis": "phase_rr",
        "volume_sensitivity_fraction_per_ml": VOLUME_SENSITIVITY_PER_ML.tolist(),
        "edv_grid_ml": EDV_GRID_ML.tolist(),
        "interpretation": "computational_sweep_not_reference_range_or_uncertainty_interval",
    },
    "assumptions": [
        "residual_extrema_are_mechanical_ed_and_es",
        "linear_constant_volume_operator",
        "equivalent_sphere_excursion_equals_single_chamber_stroke_volume",
        "edv_scenario_matches_the_same_chamber_and_cycle",
        "all_upstream_candidate_annotations_and_scale_scenarios_are_correct",
    ],
    "limitations": [
        "no_target_chamber",
        "no_mechanical_phase_operator",
        "no_calibrated_or_fem_validated_volume_sensitivity",
        "no_uncertainty_or_covariance_model",
        "phase_rr_is_not_absolute_time",
        "not_compatible_with_accepted_statuses_in_60_01_or_60_02",
    ],
    "results": results,
}
OUT_PATH.write_text(json.dumps(artifact, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

row_count = sum(item["scenario_count"] for item in results)
admissible_count = sum(item["physically_admissible_count"] for item in results)
residual_amplitudes = [item["residual_peak_to_peak"] for item in results]
summary = {
    "residual_waveforms": len(results),
    "volume_sv_ef_scenario_rows": row_count,
    "physically_admissible_rows": admissible_count,
    "physically_admissible_fraction": admissible_count / row_count,
    "residual_peak_to_peak_range": [min(residual_amplitudes), max(residual_amplitudes)],
    "strict_sv_or_ef_results": 0,
}
scenario_rows = []
for group in results:
    for row in group["volume_scenarios"]:
        scenario_rows.append({
            "Gv_fraction_per_ml": row["volume_sensitivity_fraction_per_ml"],
            "EDV_ml": row["edv_scenario_ml"],
            "SV_ml": row["sv_scenario_ml"],
            "EF_fraction": row["ef_scenario_fraction"],
            "physically_admissible": row["physically_admissible"],
        })
scenario_frame = pd.DataFrame(scenario_rows)
sensitivity_table = (
    scenario_frame.groupby("Gv_fraction_per_ml", as_index=False)
    .agg(
        scenario_rows=("physically_admissible", "size"),
        admissible_rows=("physically_admissible", "sum"),
        SV_min_ml=("SV_ml", "min"),
        SV_max_ml=("SV_ml", "max"),
        EF_min=("EF_fraction", "min"),
        EF_max=("EF_fraction", "max"),
    )
)
print("Исследовательский артефакт:", OUT_PATH.name)
display(pd.DataFrame([summary]).round(6))
print("Зависимость сценариев от неизвестного объёмного оператора Gv")
display(sensitivity_table.round(6))

Исследовательский артефакт: 60.04_equivalent_sphere_sv_ef_scenarios.json


,residual_waveforms,volume_sv_ef_scenario_rows,physically_admissible_rows,physically_admissible_fraction,residual_peak_to_peak_range,strict_sv_or_ef_results
0,54,1890,916,0.484656,"[0.0012632500760024763, 0.015584421594600044]",0


Зависимость сценариев от неизвестного объёмного оператора Gv


,Gv_fraction_per_ml,scenario_rows,admissible_rows,SV_min_ml,SV_max_ml,EF_min,EF_max
0,0.000001,270,0,1263.250076,15584.421595,5.053000,311.688432
1,0.000003,270,0,399.474749,4928.226826,1.597899,98.564537
2,0.000010,270,11,126.325008,1558.442159,0.505300,31.168843
3,0.000032,270,125,39.947475,492.822683,0.159790,9.856454
4,0.000100,270,240,12.632501,155.844216,0.050530,3.116884
5,0.000316,270,270,3.994747,49.282268,0.015979,0.985645
6,0.001000,270,270,1.263250,15.584422,0.005053,0.311688


## Как использовать таблицу

Строка с `physically_admissible=true` означает только выполнение алгебраических
условий `0 ≤ ESV ≤ EDV` внутри данного сценария. Она не делает сценарий
физиологически или метрологически обоснованным.

Все остаточные кривые заданы на шкале `phase_rr`. Эта шкала сохраняет форму
цикла, но не задаёт задержку в секундах и не локализует механические фазы.

Если УО и ФВ резко меняются при соседних значениях $G_V$, текущий сигнал не
может дать устойчивый объём без независимой калибровки оператора. Если большая
часть сетки физически недопустима, это не повод сузить сетку до удобных значений:
следует проверять шкалу сигнала, тканевую коррекцию, фазовый оператор и саму
гипотезу эквивалентной сферы.